In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Import Crew agent classes
from crewai import Agent, Task, Crew

In [3]:
import os
os.environ["CREWAI_TESTING"] = "true"

from utils import get_openai_api_key, get_serper_api_key
from langchain.tools import BaseTool
from crewai_tools import DirectoryReadTool, FileReadTool, SerperDevTool
from crewai import Agent, Task, Crew  # assuming these are imported from crewai

In [4]:
# --- Set API keys and model ---
openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [5]:
# --- Define agents ---
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour the digital landscape for potential leads. "
        "Armed with cutting-edge tools and a strategic mindset, "
        "you analyze data, trends, and interactions to unearth opportunities. "
        "Your work is crucial for meaningful engagements and driving growth."
    ),
    allow_delegation=False,
    verbose=True
)


In [6]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within CrewAI's sales department, you act as the bridge "
        "between potential clients and the solutions they need. "
        "By creating engaging, personalized messages, you guide leads "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

In [7]:
# --- Define tools ---
directory_read_tool = DirectoryReadTool(directory='./instructions')  # instance
file_read_tool = FileReadTool()                                     # instance
search_tool = SerperDevTool()                                       # instance


In [12]:
from langchain.tools import BaseTool

class SentimentAnalysisTool(BaseTool):
    name: str = "sentiment_analysis_tool"
    description: str = "Analyzes sentiment of text"

    def _run(self, text: str):
        if "bad" in text.lower():
            return "negative"
        return "positive"

    async def _arun(self, text: str):
        raise NotImplementedError("Async not implemented")

# Instantiate
sentiment_analysis_tool = SentimentAnalysisTool()


In [13]:
# --- Define tasks ---

lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector. "
        "Focus on key decision-makers, recent milestones, "
        "and potential needs that align with our offerings."
    ),
    expected_output=(
        "A detailed report on {lead_name}, "
        "including company background, key personnel, milestones, and identified needs. "
        "Suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool, sentiment_analysis_tool],
    agent=sales_rep_agent,
)



ValidationError: 1 validation error for Task
tools.3
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=SentimentAnalysisTool(), input_type=SentimentAnalysisTool]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [10]:
personalized_outreach_task = Task(
    description=(
        "Using the insights from the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign targeting {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "Address their recent {milestone} and how our solutions support their goals."
    ),
    expected_output=(
        "A series of personalized email drafts tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}. "
        "Each draft should connect our solutions to their achievements and future goals."
    ),
    # --- Key fix: include class for custom tool ---
    tools=[directory_read_tool, file_read_tool, search_tool, sentiment_analysis_tool,
    agent=lead_sales_rep_agent,
)

ValidationError: 1 validation error for Task
tools.3
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=<class '__main__.SentimentAnalysisTool'>, input_type=ModelMetaclass]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [ ]:
# --- Setup Crew ---
crew = Crew(
    agents=[sales_rep_agent, lead_sales_rep_agent],
    tasks=[lead_profiling_task.dict(), personalized_outreach_task.dict()],
    verbose=2,
    memory=True
)


In [ ]:


# --- Inputs for Crew ---
inputs = {
    "lead_name": "The AI Collective",
    "industry": "Artificial Intelligence",
    "key_decision_maker": "Chappy (Gabriel) Asel",
    "position": "Co-founder and Executive Director",
    "milestone": "Global launch, formal incorporation, scaling to 100+ chapters"
}




In [ ]:
# --- Run the Crew ---
result = crew.kickoff(inputs=inputs)
print(result)